# AML Benchmark — Part B Multi-Threshold Optimization

**Drei Threshold-Strategien** auf XGBoost Baseline Modell (Part A), kein Retraining.

## Was passiert

Auf den Validation-Scores des Part-A-XGBoost-Baseline-Modells werden drei verschiedene Threshold-Selektions-Strategien evaluiert:

1. **precision_constrained** — max F1 unter Precision >= 0.10 (operational motiviert für AML Compliance)
2. **f1_max** — max F1 (Standard-Benchmark, balancierter Trade-off)
3. **f2_max** — max F2 (Recall-gewichtet, regulatorisch motiviert)

Jede Strategie produziert einen Operating Point. Da alle drei Strategien dieselben Modell-Scores nutzen, ist **PR-AUC zwischen allen drei Strategien identisch** (Validierung im Code via Assertion bei 1e-9 Toleranz).

Vergleichs-Referenz: Part A XGBoost Baseline beim F1-optimalen Threshold (~0.0405), nicht beim Default 0.5.

## Run-Konfiguration

**Branch:** `feature/part-b-multi-threshold`

**Verwendeter Baseline-Run:** `xgboost__baseline__p001__20260404_143052` (alle drei Baseline-Runs sind bit-identisch, da Baseline invariant zu target_prevalence ist — daher reicht einer als Repräsentant)

**Resultierende Datenpunkte:** 1 Modell × 3 Strategien = 3 Operating Points + 1 Reference-Zeile = 4 Zeilen in Table 5

**Laufzeit:** ~5-10 Minuten (kein Training, nur Threshold-Suche auf gespeicherten Modell-Scores)

**Outputs:**
- `outputs/part_b_thresholds/<run_id>/<strategy>/` (pro Strategie: metrics_val/test, threshold_info.json)
- `results/part_b_multi_threshold_summary.json` (Aggregat aller drei Strategien)
- `results/tables/table5_part_b_multi_threshold.{csv,md}` (Vergleichstabelle)

**Voraussetzung:** Part A v2 abgeschlossen. Feature Cache + Runs auf Drive unter `large_run_v2_20260407_1904`. `threshold_info.json` muss für den verwendeten Baseline-Run existieren.

## Schritt 1 — Google Drive mounten

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive gemountet.')

## Schritt 2 — RAM und GPU pruefen

In [ ]:
import psutil
ram = psutil.virtual_memory()
print(f'Gesamt-RAM : {ram.total / 1e9:.1f} GB')
print(f'Freier RAM : {ram.available / 1e9:.1f} GB')
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null || echo 'Kein GPU (fuer Threshold-Optimierung nicht noetig)'

## Schritt 3 — Projektcode klonen (Branch: feature/part-b-multi-threshold)

In [ ]:
import os
from pathlib import Path

!git clone -b feature/part-b-multi-threshold https://github.com/fdrmic/classimbalance.git /content/classimbalance

PROJECT_DIR = Path('/content/classimbalance')
os.chdir(PROJECT_DIR)
print('Arbeitsverzeichnis:', os.getcwd())
print('Branch:', os.popen('git branch --show-current').read().strip())
print('Letzter Commit:', os.popen('git log -1 --oneline').read().strip())

## Schritt 4 — Dependencies installieren

In [ ]:
import os
os.chdir('/content/classimbalance')
!pip install -e . -q
!pip install -r requirements.txt -q
print('Installation abgeschlossen.')
print('WICHTIG: Runtime jetzt neu starten (Laufzeit > Sitzung neu starten), danach mit Schritt 5 weitermachen.')

## Schritt 5 — Verifizieren (NACH Runtime-Neustart ausfuehren)

Prueft, dass der neue Multi-Threshold-Code geladen werden kann.

In [ ]:
import os
os.chdir('/content/classimbalance')

# Import des Multi-Threshold Optimizers
from aml_benchmark.experiments.threshold_optimizer import (
    run_threshold_optimization,
    VALID_STRATEGIES,
    PREFERRED_RUN_NAME,
)

print('OK — threshold_optimizer geladen')
print(f'Verfuegbare Strategien : {VALID_STRATEGIES}')
print(f'Bevorzugter Baseline-Run: {PREFERRED_RUN_NAME}')

## Schritt 5b — Dry-Run zur Code-Validierung

Fuehrt eine Mock-Optimierung mit zufaelligen Scores durch. Bestaetigt, dass:
- alle drei Strategien lauffaehig sind
- das Output-Schema vollstaendig ist
- die PR-AUC-Invarianz-Assertion bei 1e-9 haelt

In [ ]:
!python -m aml_benchmark.experiments.threshold_optimizer --dry-run

## Schritt 6 — paths_large_v2.yaml fuer Colab konfigurieren

In [ ]:
import yaml
from pathlib import Path

PROJECT_DIR = Path(os.getcwd())
DRIVE_DIR   = Path('/content/drive/MyDrive/aml_data')

config_path = PROJECT_DIR / 'configs' / 'paths_large_v2.yaml'

with open(config_path) as f:
    cfg = yaml.safe_load(f)

cfg['raw_dir']         = str(DRIVE_DIR)
cfg['processed_dir']   = str(PROJECT_DIR / 'data' / 'processed_v2')
cfg['splits_dir']      = str(PROJECT_DIR / 'data' / 'splits_v2')
cfg['outputs_dir']     = str(PROJECT_DIR / 'outputs' / 'runs_v2')
cfg['leaderboard_dir'] = str(PROJECT_DIR / 'outputs' / 'leaderboard_v2')

with open(config_path, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

print('paths_large_v2.yaml aktualisiert:')
print(yaml.dump(cfg, default_flow_style=False))

## Schritt 7 — Part A Runs und Feature Cache von Drive laden

Nur Splits und Part A Runs laden, kein Re-Processing.

In [ ]:
import shutil
from pathlib import Path

PROJECT_DIR = Path(os.getcwd())
backup = Path('/content/drive/MyDrive/aml_results/large_run_v2_20260407_1904')

for folder, dst_path in [
    ('splits', PROJECT_DIR / 'data' / 'splits_v2'),
    ('runs',   PROJECT_DIR / 'outputs' / 'runs_v2'),
]:
    src = backup / folder
    if src.exists():
        dst_path.mkdir(parents=True, exist_ok=True)
        shutil.copytree(src, dst_path, dirs_exist_ok=True)
        print(f'Geladen: {folder}/')
    else:
        print(f'Nicht vorhanden: {folder}/')

# Ongoing Runs dazuladen falls vorhanden
ongoing = Path('/content/drive/MyDrive/aml_results/large_run_v2_ongoing')
runs_src = ongoing / 'runs'
runs_dst = PROJECT_DIR / 'outputs' / 'runs_v2'
if runs_src.exists():
    shutil.copytree(runs_src, runs_dst, dirs_exist_ok=True)
    print('Ongoing runs dazugeladen.')

# Splits pruefen
splits_dir = PROJECT_DIR / 'data' / 'splits_v2'
for f in ['val.parquet', 'test.parquet', 'val_features_v2.parquet', 'test_features_v2.parquet']:
    p = splits_dir / f
    status = f'{p.stat().st_size / 1e9:.2f} GB' if p.exists() else 'FEHLT'
    print(f'  {f}: {status}')

# XGBoost Baseline Runs pruefen
runs_dir = PROJECT_DIR / 'outputs' / 'runs_v2'
baseline_runs = [r for r in runs_dir.iterdir()
                 if r.is_dir() and r.name.startswith('xgboost__baseline__')
                 and (r / 'model.pkl').exists()]
print(f'\nXGBoost Baseline Runs gefunden: {len(baseline_runs)}')
for r in sorted(baseline_runs):
    print(f'  {r.name}')

## Schritt 7b — threshold_info.json fuer den verwendeten Run pruefen

Der Multi-Threshold-Optimizer braucht den F1-optimalen Threshold aus Part A als Vergleichs-Referenz.
Bevorzugter Run: `xgboost__baseline__p001__20260404_143052`.

Falls die Datei fehlt, bricht der Run ab — dann muss `re_evaluate.py` zuerst laufen.

In [ ]:
import json
from pathlib import Path
from aml_benchmark.experiments.threshold_optimizer import PREFERRED_RUN_NAME

PROJECT_DIR = Path(os.getcwd())
runs_dir = PROJECT_DIR / 'outputs' / 'runs_v2'

preferred_run = runs_dir / PREFERRED_RUN_NAME
thresh_path   = preferred_run / 'threshold_info.json'

print(f'Bevorzugter Baseline-Run : {PREFERRED_RUN_NAME}')
print(f'Pfad existiert           : {preferred_run.exists()}')
print(f'model.pkl existiert      : {(preferred_run / "model.pkl").exists()}')
print(f'threshold_info.json      : {thresh_path.exists()}')
print()

if thresh_path.exists():
    with open(thresh_path) as f:
        ti = json.load(f)
    print('Inhalt threshold_info.json:')
    print(f"  optimal_threshold   : {ti.get('optimal_threshold', 'N/A')}")
    print(f"  threshold_criterion : {ti.get('threshold_criterion', 'N/A')}")
    print()
    print('OK — Multi-Threshold-Run kann gestartet werden.')
else:
    print('FEHLT — threshold_info.json muss zuerst erzeugt werden.')
    print('Optionen:')
    print('  1) re_evaluate.py auf den Run laufen lassen')
    print('  2) Anderen Baseline-Run waehlen, der die Datei hat')

## Schritt 8 — Multi-Threshold Optimization ausfuehren

Drei Strategien, ein Baseline-Run, kein Training (~5-10 Minuten).

Alle Hyperparameter explizit gesetzt, damit der Run nachvollziehbar ist:
- `--strategies precision_constrained f1_max f2_max` — alle drei
- `--precision-constraint 0.10` — fuer precision_constrained
- `--lambda-fp 0.05` — Tiebreak-Utility fuer precision_constrained
- `--n-dense 1000` — Threshold-Grid-Aufloesung (Quantil-basiert)

In [ ]:
!python -m aml_benchmark.experiments.threshold_optimizer \
    --paths configs/paths_large_v2.yaml \
    --strategies precision_constrained f1_max f2_max \
    --precision-constraint 0.10 \
    --lambda-fp 0.05 \
    --n-dense 1000

## Schritt 9 — Ergebnisse anzeigen

Liest `results/part_b_multi_threshold_summary.json` und zeigt eine Vergleichstabelle aller drei Strategien plus Part-A-Reference.

In [ ]:
import json
import pandas as pd
from pathlib import Path

PROJECT_DIR = Path(os.getcwd())
summary_path = PROJECT_DIR / 'results' / 'part_b_multi_threshold_summary.json'

if not summary_path.exists():
    print('Summary-Datei nicht gefunden. Schritt 8 ausfuehren.')
else:
    with open(summary_path) as f:
        summary = json.load(f)

    print('=' * 78)
    print('PART B MULTI-THRESHOLD RESULTS')
    print('=' * 78)
    print(f"Selected Run        : {summary.get('selected_run_id', 'N/A')}")
    print(f"Strategies          : {summary.get('strategies', [])}")
    print()
    print(f"Note                : {summary.get('note', '')}")
    print()

    # PR-AUC Invariance Check
    inv = summary.get('pr_auc_invariance', {})
    print(f"PR-AUC Invariance Check:")
    print(f"  Spread (max - min)  : {inv.get('spread', 'N/A')}")
    print(f"  Tolerance           : {inv.get('tolerance', 'N/A')}")
    print(f"  Status              : {inv.get('status', 'N/A')}")
    print()

    # Part A Reference
    ref = summary.get('part_a_reference', {})
    print(f"Part A Baseline (F1-optimal Reference):")
    print(f"  Threshold : {ref.get('threshold', 'N/A')}")
    print(f"  Precision : {ref.get('precision', 'N/A')}")
    print(f"  Recall    : {ref.get('recall', 'N/A')}")
    print(f"  F1        : {ref.get('f1', 'N/A')}")
    print(f"  F2        : {ref.get('f2', 'N/A')}")
    print(f"  FP        : {ref.get('fp', 'N/A')}")
    print(f"  PR-AUC    : {ref.get('pr_auc', 'N/A')}")
    print()

    # Strategy Comparison
    print('Strategy Comparison (Test Set):')
    print('-' * 78)
    rows = []
    rows.append({
        'Strategy': 'Part A Baseline (F1-opt)',
        'Threshold': f"{ref.get('threshold', 0):.6f}",
        'Precision': f"{ref.get('precision', 0):.4f}",
        'Recall':    f"{ref.get('recall', 0):.4f}",
        'F1':        f"{ref.get('f1', 0):.4f}",
        'F2':        f"{ref.get('f2', 0):.4f}",
        'FP':        f"{int(ref.get('fp', 0)):,}",
        'FP_delta':  '0',
        'F1_delta':  '0.0000',
    })
    for rec in summary.get('records', []):
        rows.append({
            'Strategy':  rec['strategy_type'],
            'Threshold': f"{rec['threshold_value']:.6f}",
            'Precision': f"{rec['test_precision']:.4f}",
            'Recall':    f"{rec['test_recall']:.4f}",
            'F1':        f"{rec['test_f1']:.4f}",
            'F2':        f"{rec['test_f2']:.4f}",
            'FP':        f"{int(rec['test_fp']):,}",
            'FP_delta':  f"{int(rec['delta_fp_vs_part_a_f1opt']):+,}",
            'F1_delta':  f"{rec['delta_f1_vs_part_a_f1opt']:+.4f}",
        })
    df = pd.DataFrame(rows)
    print(df.to_string(index=False))
    print('=' * 78)

    # PR-AUC Bestaetigung
    pr_aucs = [rec['test_pr_auc'] for rec in summary.get('records', [])]
    if pr_aucs:
        print(f'\nPR-AUC test ueber alle drei Strategien:')
        for rec in summary.get('records', []):
            print(f"  {rec['strategy_type']:<22} : {rec['test_pr_auc']:.10f}")
        print(f'  → identisch wie erwartet (PR-AUC ist threshold-invariant)')

## Schritt 9b — Vergleichstabelle (Table 5) generieren

Erzeugt `results/tables/table5_part_b_multi_threshold.csv` und `.md` aus dem Summary-File.

In [ ]:
!python -m aml_benchmark.analysis.results_tables

table5_md  = PROJECT_DIR / 'results' / 'tables' / 'table5_part_b_multi_threshold.md'
table5_csv = PROJECT_DIR / 'results' / 'tables' / 'table5_part_b_multi_threshold.csv'

print()
print(f'Table 5 CSV: {table5_csv} (existiert: {table5_csv.exists()})')
print(f'Table 5 MD : {table5_md} (existiert: {table5_md.exists()})')

if table5_md.exists():
    print()
    print('=== Table 5 Markdown ===')
    print(table5_md.read_text())

## Schritt 10 — Ergebnisse auf Drive sichern

Sichert vor dem Session-Ende:
- `outputs/part_b_thresholds/` (pro Strategie: metrics + threshold_info)
- `results/part_b_multi_threshold_summary.json`
- `results/tables/table5_part_b_multi_threshold.{csv,md}`

In [ ]:
import shutil, datetime
from pathlib import Path

PROJECT_DIR = Path(os.getcwd())
ts = datetime.datetime.now().strftime('%Y%m%d_%H%M')
backup_dir = Path(f'/content/drive/MyDrive/aml_results/part_b_multi_run_{ts}')
backup_dir.mkdir(parents=True, exist_ok=True)

# 1) outputs/part_b_thresholds/
src = PROJECT_DIR / 'outputs' / 'part_b_thresholds'
if src.exists():
    shutil.copytree(src, backup_dir / 'part_b_thresholds', dirs_exist_ok=True)
    print(f'Gesichert: outputs/part_b_thresholds/')
else:
    print('Nicht gefunden: outputs/part_b_thresholds/ — Schritt 8 ausfuehren.')

# 2) results/part_b_multi_threshold_summary.json
summary_src = PROJECT_DIR / 'results' / 'part_b_multi_threshold_summary.json'
if summary_src.exists():
    (backup_dir / 'results').mkdir(parents=True, exist_ok=True)
    shutil.copy(summary_src, backup_dir / 'results' / 'part_b_multi_threshold_summary.json')
    print(f'Gesichert: results/part_b_multi_threshold_summary.json')

# 3) Table 5 (CSV + MD)
tables_src = PROJECT_DIR / 'results' / 'tables'
if tables_src.exists():
    (backup_dir / 'results').mkdir(parents=True, exist_ok=True)
    for f in tables_src.glob('table5_part_b_multi_threshold.*'):
        shutil.copy(f, backup_dir / 'results' / f.name)
        print(f'Gesichert: {f.name}')

print(f'\nBackup abgeschlossen: {backup_dir}')

---
## Hinweise zum Run

**Kein neues Training:** Nutzt das gespeicherte XGBoost Baseline Modell aus Part A, lädt nur model.pkl und ruft predict_proba auf.

**Schritt 5 nach pip install:** Runtime neu starten, dann Schritt 5 + 5b ausfuehren bevor weitergegangen wird.

**PR-AUC Invariance:** Alle drei Strategien produzieren bit-identische PR-AUC, weil sie auf demselben Score-Vektor operieren. Der Code prueft das mit Toleranz 1e-9 — bei Verletzung bricht der Run mit RuntimeError ab. Das ist ein Schutz gegen versehentliches Retraining oder Score-Korruption.

**Vergleichs-Referenz:** Part A Baseline beim F1-optimalen Threshold (~0.0405), nicht beim Default 0.5. Begruendung: 0.5 ist kein Operating Point, den ein AML-Compliance-Team realistisch verwenden wuerde — der Vergleich gegen 0.5 wuerde Verbesserungen kuenstlich aufblaehen.

**Trade-off statt Sieger:** Die drei Strategien produzieren unterschiedliche Operating Points im Precision-Recall-Raum. Es gibt keinen Sieger — die Wahl haengt von operationalen Constraints ab. Fuer AML-Compliance mit fixer Investigations-Kapazitaet ist precision_constrained typischerweise die operativ angemessene Wahl. f1_max und f2_max dienen als wissenschaftliche Vergleichs-Anker.

**Session beenden:** Nach Schritt 10 — Laufzeit trennen und loeschen, damit Drive-Backup vollstaendig synchronisiert.